In [58]:
import pandas as pd
import numpy as np
from pathlib import Path

In [59]:
# Find the project folder without a personal computer path.
REPO_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data" / "raw" / "home-credit-default-risk").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Run this notebook inside the project and extract Kaggle CSVs into "
        "data/raw/home-credit-default-risk/."
    )

DATA_RAW = REPO_ROOT / "data" / "raw" / "home-credit-default-risk"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
DATA_INTERIM = REPO_ROOT / "data" / "interim"


In [60]:
BUREAU_PATH = DATA_RAW / "bureau.csv"
BBALANCE_PATH = DATA_RAW / "bureau_balance.csv"

In [61]:
df = pd.read_csv(BBALANCE_PATH)

In [62]:
df.head()

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [63]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Dtype
---  ------          -----
 0   SK_ID_BUREAU    int64
 1   MONTHS_BALANCE  int64
 2   STATUS          str  
dtypes: int64(2), str(1)
memory usage: 650.9 MB


In [64]:
df.shape

(27299925, 3)

In [65]:
df["STATUS"].value_counts(dropna=False)

STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64

For Status column values:
``` 
C means closed --> يعني الكارت اتقفل في الشهر دا
X means status unknown -->  مفيش معلومات عن الكارت
0 means no DPD (Days Past Due) --> لم يتم الابلاغ عن اي ايام متأخرة
1 means maximal did during month between 1-30 --> في سنة تأخير
2 means DPD (Days Past Due) 31-60 --> في شوية تأخير
� 5 means DPD 120+ or sold or written off --> لا التأخير كان كبير هنا
```

In [66]:
grouped = df.groupby('SK_ID_BUREAU')

In [67]:
bb_features = grouped.agg(bb_month_count=('MONTHS_BALANCE', 'count')).reset_index()

Here we will make a col named: bb_month_count, it will express the number of months for every user.

In [68]:
bb_features.head()

,SK_ID_BUREAU,bb_month_count
0,5001709,97
1,5001710,83
2,5001711,4
3,5001712,19
4,5001713,22


In [69]:
bb_features.shape

(817395, 2)

There are total 817,395 users in bureau_balance.csv

In [70]:
bb_features = grouped.agg(
    bb_months_count=('MONTHS_BALANCE', 'count'),
    bb_delinquent_months=('STATUS', lambda x: x.isin(["1", "2", "3", "4", "5"]).sum())
).reset_index()

In [71]:
bb_features['bb_delinquent_rate'] = bb_features['bb_delinquent_months'] / bb_features['bb_months_count']

- Delinquent = متأخر في سداد الديون أو الأقساط.
- bb_delinquent_months = متأخر في السداد كام شهر
- bb_delinquent_rate = نسبة التأخر كام في اجمالي المدة

In [72]:
bb_features.sample(5)

,SK_ID_BUREAU,bb_months_count,bb_delinquent_months,bb_delinquent_rate
596595,6396934,2,0,0.0
308715,5918448,18,0,0.0
69515,5182828,10,0,0.0
335391,5953929,47,0,0.0
470190,6153897,14,0,0.0


In [73]:
bb_features.drop(columns='SK_ID_BUREAU').describe()

,bb_months_count,bb_delinquent_months,bb_delinquent_rate
count,817395.000000,817395.000000,817395.000000
mean,33.398693,0.419556,0.013772
std,25.794666,2.275379,0.059165
min,1.000000,0.000000,0.000000
25%,13.000000,0.000000,0.000000
50%,26.000000,0.000000,0.000000
75%,48.000000,0.000000,0.000000
max,97.000000,97.000000,1.000000


In [74]:
bb_features["bb_max_status"] = grouped["STATUS"].apply(
    lambda x: pd.to_numeric(x[x.isin(["0", "1", "2", "3", "4", "5"])]).max()
).values

هنا بنعرف اقصي فترة تأخير في السداد

In [75]:
bb_features['bb_max_status'].value_counts(dropna=False).sort_index()

bb_max_status
0.0    583763
1.0     87055
2.0      7841
3.0      2077
4.0      1077
5.0      5214
NaN    130368
Name: count, dtype: int64

In [76]:
bb_features["bb_distinct_delinquent_statuses"] = grouped["STATUS"].apply(
    lambda x: x[x.isin(["1", "2", "3", "4", "5"])].nunique()
).values

In [77]:
bb_features["bb_distinct_delinquent_statuses"].value_counts().sort_index()

bb_distinct_delinquent_statuses
0    714131
1     89861
2      7124
3      1992
4      1348
5      2939
Name: count, dtype: int64

In [78]:
latest_status = (
    df.loc[df.groupby('SK_ID_BUREAU')['MONTHS_BALANCE'].idxmax(), ['SK_ID_BUREAU', 'STATUS']]
    .rename(columns={'STATUS': 'bb_latest_status'})
)

bb_features = bb_features.merge(
    latest_status,
    on="SK_ID_BUREAU",
    how="left",
    validate="one_to_one"
)

In [79]:
bb_features.head()

,SK_ID_BUREAU,bb_months_count,bb_delinquent_months,bb_delinquent_rate,bb_max_status,bb_distinct_delinquent_statuses,bb_latest_status
0,5001709,97,0,0.0,NaN,0,C
1,5001710,83,0,0.0,0.0,0,C
2,5001711,4,0,0.0,0.0,0,X
3,5001712,19,0,0.0,0.0,0,C
4,5001713,22,0,0.0,NaN,0,X


In [80]:
bb_features["bb_history_oldest_month"] = grouped["MONTHS_BALANCE"].min().values

عشان نعرف الاكونت دا بقالو قد ايه

In [81]:
bb_features["bb_has_delinquency"] = (
    bb_features["bb_delinquent_months"] > 0
).astype("int8")

In [82]:
bb_features.head()

,SK_ID_BUREAU,bb_months_count,bb_delinquent_months,bb_delinquent_rate,bb_max_status,bb_distinct_delinquent_statuses,bb_latest_status,bb_history_oldest_month,bb_has_delinquency
0,5001709,97,0,0.0,NaN,0,C,-96,0
1,5001710,83,0,0.0,0.0,0,C,-82,0
2,5001711,4,0,0.0,0.0,0,X,-3,0
3,5001712,19,0,0.0,0.0,0,C,-18,0
4,5001713,22,0,0.0,NaN,0,X,-21,0


In [83]:
bb_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 817395 entries, 0 to 817394
Data columns (total 9 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   SK_ID_BUREAU                     817395 non-null  int64  
 1   bb_months_count                  817395 non-null  int64  
 2   bb_delinquent_months             817395 non-null  int64  
 3   bb_delinquent_rate               817395 non-null  float64
 4   bb_max_status                    687027 non-null  float64
 5   bb_distinct_delinquent_statuses  817395 non-null  int64  
 6   bb_latest_status                 817395 non-null  str    
 7   bb_history_oldest_month          817395 non-null  int64  
 8   bb_has_delinquency               817395 non-null  int8   
dtypes: float64(2), int64(5), int8(1), str(1)
memory usage: 51.5 MB


In [84]:
bb_features.shape

(817395, 9)

### Bureau Balance Feautres
- SK_ID_BUREAU --> External account ID
- bb_months_count-->Number of observed months
- bb_delinquent_months --> Number of delinquent months
- bb_delinquent_rate --> Fraction of delinquent months
- bb_max_status --> Worst numeric status
- bb_distinct_delinquent_statuses --> Number of different delinquency levels
- bb_latest_status --> Most recent status
- bb_history_oldest_month --> Oldest observed month
- bb_has_delinquency --> Whether delinquency ever occurred

In [85]:
print('Total rows:', len(bb_features))
print('Unique accounts:', bb_features['SK_ID_BUREAU'].nunique())
print('Duplicates IDs:', bb_features['SK_ID_BUREAU'].duplicated().sum())

Total rows: 817395
Unique accounts: 817395
Duplicates IDs: 0


## Merge Bureau Balance Features with Bureau.csv

In [86]:
bureau = pd.read_csv(BUREAU_PATH)

In [87]:
print("Shape:", bureau.shape)
print("Unique accounts:", bureau["SK_ID_BUREAU"].nunique())

bureau.head()

Shape: (1716428, 17)
Unique accounts: 1716428


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [88]:
bureau_enriched = bureau.merge(
    bb_features,
    on='SK_ID_BUREAU',
    how='left',
    validate='one_to_one'
)

In [89]:
print("Original bureau shape:", bureau.shape)
print("Enriched bureau shape:", bureau_enriched.shape)

Original bureau shape: (1716428, 17)
Enriched bureau shape: (1716428, 25)


In [90]:
bureau_enriched.sample(5)

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,DAYS_CREDIT_UPDATE,AMT_ANNUITY,bb_months_count,bb_delinquent_months,bb_delinquent_rate,bb_max_status,bb_distinct_delinquent_statuses,bb_latest_status,bb_history_oldest_month,bb_has_delinquency
1030396,107765,6218768,Active,currency 1,-346,0,385.0,NaN,0.00,0,...,-12,1543.995,12.0,0.0,0.0,0.0,0.0,0,-11.0,0.0
1187222,180559,6805944,Active,currency 1,-261,0,10699.0,NaN,0.00,0,...,-32,135000.000,9.0,0.0,0.0,0.0,0.0,X,-8.0,0.0
405354,120123,6419770,Sold,currency 1,-1985,0,-1801.0,NaN,NaN,0,...,-1967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
859275,196500,6040591,Active,currency 1,-1024,0,802.0,NaN,NaN,0,...,-15,26266.500,34.0,0.0,0.0,0.0,0.0,0,-33.0,0.0
963240,379139,5474422,Active,currency 1,-419,0,679.0,NaN,5528.61,0,...,-22,NaN,14.0,0.0,0.0,0.0,0.0,0,-13.0,0.0


In [91]:
print(
    "Accounts without monthly history:",
    bureau_enriched["bb_months_count"].isna().sum()
)

print(
    "Accounts with monthly history:",
    bureau_enriched["bb_months_count"].notna().sum()
)

Accounts without monthly history: 942074
Accounts with monthly history: 774354


there is accounts that we have no monthly history for it.

### We Will Group by Applicant ID

In [92]:
bureau_grouped = bureau_enriched.groupby('SK_ID_CURR')

In [93]:
print("Applicants:", bureau_grouped.ngroups)
print("Original bureau accounts:", len(bureau_enriched))

Applicants: 305811
Original bureau accounts: 1716428


In [94]:

bureau_features = bureau_grouped.agg(
    bureau_total_accounts=("SK_ID_BUREAU", "count"),

    bureau_active_accounts=(
        "CREDIT_ACTIVE",
        lambda x: (x == "Active").sum()
    ),

    bureau_closed_accounts=(
        "CREDIT_ACTIVE",
        lambda x: (x == "Closed").sum()
    ),

    bureau_consumer_credit_accounts=(
        "CREDIT_TYPE",
        lambda x: (x == "Consumer credit").sum()
    ),

    bureau_credit_card_accounts=(
        "CREDIT_TYPE",
        lambda x: (x == "Credit card").sum()
    ),

    bureau_credit_sum_total=("AMT_CREDIT_SUM", "sum"),
    bureau_credit_sum_mean=("AMT_CREDIT_SUM", "mean"),
    bureau_credit_sum_max=("AMT_CREDIT_SUM", "max"),

    # Preserve NaN when all debt values are missing
    bureau_debt_total=(
        "AMT_CREDIT_SUM_DEBT",
        lambda x: x.sum(min_count=1)
    ),
    bureau_debt_mean=("AMT_CREDIT_SUM_DEBT", "mean"),
    bureau_debt_max=("AMT_CREDIT_SUM_DEBT", "max"),

    bureau_overdue_total=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    bureau_overdue_max=("AMT_CREDIT_SUM_OVERDUE", "max"),

    bureau_overdue_accounts=(
        "AMT_CREDIT_SUM_OVERDUE",
        lambda x: (x > 0).sum()
    ),

    bureau_days_overdue_max=("CREDIT_DAY_OVERDUE", "max"),
    bureau_days_overdue_mean=("CREDIT_DAY_OVERDUE", "mean"),

    bureau_missing_debt_accounts=(
        "AMT_CREDIT_SUM_DEBT",
        lambda x: x.isna().sum()
    ),

    bureau_accounts_with_debt=(
        "AMT_CREDIT_SUM_DEBT",
        lambda x: (x > 0).sum()
    ),

    # Credit history timing
    bureau_days_credit_min=("DAYS_CREDIT", "min"),
    bureau_days_credit_max=("DAYS_CREDIT", "max"),
    bureau_days_credit_mean=("DAYS_CREDIT", "mean")

).reset_index()


## 2. External Account Counts

These features describe how many external credit accounts the applicant has.

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `bureau_total_accounts` | Total number of external credit accounts belonging to the applicant. | إجمالي عدد حسابات الائتمان الخارجية. |
| `bureau_active_accounts` | Number of accounts currently marked as Active in the bureau data. | عدد الحسابات النشطة. |
| `bureau_closed_accounts` | Number of accounts marked as Closed. | عدد الحسابات المغلقة. |
| `bureau_consumer_credit_accounts` | Number of external Consumer Credit accounts. | عدد حسابات الائتمان الاستهلاكي. |
| `bureau_credit_card_accounts` | Number of external credit card accounts. | عدد حسابات بطاقات الائتمان. |

Note: Active and Closed accounts might not add up to the total because other statuses exist, such as Sold and Bad debt.

## 3. Credit Amount Features

These features describe the credit amounts reported across the applicant's external accounts.

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `bureau_credit_sum_total` | Sum of the reported credit amounts across all external accounts. | إجمالي مبالغ الائتمان المسجلة. |
| `bureau_credit_sum_mean` | Average reported credit amount per external account. | متوسط مبلغ الائتمان لكل حساب. |
| `bureau_credit_sum_max` | Largest reported credit amount among the applicant's accounts. | أكبر مبلغ ائتمان مسجل بين الحسابات. |

## 4. Outstanding Debt Features

These features describe the remaining debt reported by external credit providers.

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `bureau_debt_total` | Sum of reported outstanding debt across the applicant's accounts. | إجمالي الديون المتبقية المسجلة. |
| `bureau_debt_mean` | Average reported outstanding debt per account, considering available values. | متوسط الديون المتبقية لكل حساب. |
| `bureau_debt_max` | Highest outstanding debt reported for one account. | أكبر قيمة دين متبقٍ في حساب واحد. |


## 5. Overdue Features

These features describe overdue amounts and reported overdue days.

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `bureau_overdue_total` | Total reported overdue amount across all external accounts. | إجمالي المبالغ المتأخرة في السداد. |
| `bureau_overdue_max` | Highest overdue amount reported for one account. | أكبر مبلغ متأخر في حساب واحد. |
| `bureau_overdue_accounts` | Number of accounts with a positive reported overdue amount. | عدد الحسابات التي لديها مبلغ متأخر أكبر من صفر. |
| `bureau_days_overdue_max` | Maximum reported number of overdue days across the applicant's accounts. | أكبر عدد أيام تأخير مسجل. |
| `bureau_days_overdue_mean` | Average reported overdue days across the applicant's accounts. | متوسط عدد أيام التأخير. |

## 6. Debt Data Quality Features

These features help us understand how complete the reported debt information is.

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `bureau_missing_debt_accounts` | Number of accounts where the outstanding debt value is missing. | عدد الحسابات التي لا تتوفر لها بيانات الدين المتبقي. |
| `bureau_accounts_with_debt` | Number of accounts with a positive reported outstanding debt. | عدد الحسابات التي لديها دين متبقٍ أكبر من صفر. |

In [95]:
bureau_features["bureau_accounts_with_debt_ratio"] = (
    bureau_features["bureau_accounts_with_debt"] /
    bureau_features["bureau_total_accounts"]
)

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `bureau_debt_to_credit_ratio` | Total reported debt divided by total reported credit. Indicates the relationship between outstanding debt and reported credit. | نسبة إجمالي الدين المتبقي إلى إجمالي الائتمان المسجل. |

A ratio of `0.4` means reported debt is 40% of the reported credit total. This is a portfolio-level indicator, not necessarily the exact repayment percentage.

In [96]:
bureau_features.head()

,SK_ID_CURR,bureau_total_accounts,bureau_active_accounts,bureau_closed_accounts,bureau_consumer_credit_accounts,bureau_credit_card_accounts,bureau_credit_sum_total,bureau_credit_sum_mean,bureau_credit_sum_max,bureau_debt_total,...,bureau_overdue_max,bureau_overdue_accounts,bureau_days_overdue_max,bureau_days_overdue_mean,bureau_missing_debt_accounts,bureau_accounts_with_debt,bureau_days_credit_min,bureau_days_credit_max,bureau_days_credit_mean,bureau_accounts_with_debt_ratio
0,100001,7,3,4,7,0,1453365.000,207623.571429,378000.0,596686.5,...,0.0,0,0,0.0,0,3,-1572,-49,-735.000000,0.428571
1,100002,8,2,6,4,4,865055.565,108131.945625,450000.0,245781.0,...,0.0,0,0,0.0,3,1,-1437,-103,-874.000000,0.125000
2,100003,4,1,3,2,2,1017400.500,254350.125000,810000.0,0.0,...,0.0,0,0,0.0,0,0,-2586,-606,-1400.750000,0.000000
3,100004,2,0,2,2,0,189037.800,94518.900000,94537.8,0.0,...,0.0,0,0,0.0,0,0,-1326,-408,-867.000000,0.000000
4,100005,3,2,1,2,1,657126.000,219042.000000,568800.0,568408.5,...,0.0,0,0,0.0,0,2,-373,-62,-190.666667,0.666667


In [97]:
bureau_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 305811 entries, 0 to 305810
Data columns (total 23 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   SK_ID_CURR                       305811 non-null  int64  
 1   bureau_total_accounts            305811 non-null  int64  
 2   bureau_active_accounts           305811 non-null  int64  
 3   bureau_closed_accounts           305811 non-null  int64  
 4   bureau_consumer_credit_accounts  305811 non-null  int64  
 5   bureau_credit_card_accounts      305811 non-null  int64  
 6   bureau_credit_sum_total          305811 non-null  float64
 7   bureau_credit_sum_mean           305809 non-null  float64
 8   bureau_credit_sum_max            305809 non-null  float64
 9   bureau_debt_total                297439 non-null  float64
 10  bureau_debt_mean                 297439 non-null  float64
 11  bureau_debt_max                  297439 non-null  float64
 12  bureau_overdu

In [98]:
bb_grouped = bureau_enriched.groupby("SK_ID_CURR")

In [99]:

bb_applicant_features = bb_grouped.agg(
    bureau_bb_accounts=("SK_ID_BUREAU", "count"),

    bureau_bb_accounts_with_history=(
        "bb_months_count",
        "count"
    ),

    bureau_bb_delinquent_accounts=(
        "bb_has_delinquency",
        lambda x: x.sum(min_count=1)
    ),

    bureau_bb_delinquent_months=(
        "bb_delinquent_months",
        lambda x: x.sum(min_count=1)
    ),

    bureau_bb_delinquent_rate_mean=(
        "bb_delinquent_rate",
        "mean"
    ),

    bureau_bb_max_status=(
        "bb_max_status",
        "max"
    ),

    bureau_bb_months_mean=(
        "bb_months_count",
        "mean"
    )

).reset_index()

## 7. Monthly Credit History Features

These features come from `bureau_balance.csv`, after aggregating monthly records to individual credit accounts and then to applicants.

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `bureau_bb_accounts` | Total number of external credit accounts considered for the applicant. | إجمالي عدد الحسابات الخارجية. |
| `bureau_bb_delinquent_accounts` | Number of accounts that experienced at least one delinquent month in their available monthly history. | عدد الحسابات التي شهدت تأخيرًا في السداد مرة واحدة على الأقل. |
| `bureau_bb_delinquent_months` | Total number of delinquent monthly records across the applicant's accounts. | إجمالي عدد الأشهر التي تم فيها تسجيل تأخير في السداد. |
| `bureau_bb_delinquent_rate_mean` | Average delinquency rate across accounts with available monthly history. | متوسط نسبة أشهر التأخير بين الحسابات التي لها سجل شهري متاح. |
| `bureau_bb_max_status` | Worst numeric delinquency status observed across the applicant's external accounts. | أعلى مستوى تأخير مسجل بين الحسابات. |
| `bureau_bb_months_mean` | Average number of observed monthly records per external account with monthly history. | متوسط عدد الأشهر المسجلة لكل حساب لديه سجل شهري. |

In [100]:
bb_applicant_features.head()

,SK_ID_CURR,bureau_bb_accounts,bureau_bb_accounts_with_history,bureau_bb_delinquent_accounts,bureau_bb_delinquent_months,bureau_bb_delinquent_rate_mean,bureau_bb_max_status,bureau_bb_months_mean
0,100001,7,7,1.0,1.0,0.007519,1.0,24.571429
1,100002,8,8,6.0,27.0,0.255682,1.0,13.750000
2,100003,4,0,NaN,NaN,NaN,NaN,NaN
3,100004,2,0,NaN,NaN,NaN,NaN,NaN
4,100005,3,3,0.0,0.0,0.000000,0.0,7.000000


In [101]:
bb_applicant_features.describe()

,SK_ID_CURR,bureau_bb_accounts,bureau_bb_accounts_with_history,bureau_bb_delinquent_accounts,bureau_bb_delinquent_months,bureau_bb_delinquent_rate_mean,bureau_bb_max_status,bureau_bb_months_mean
count,305811.000000,305811.000000,305811.000000,134542.000000,134542.000000,134542.000000,130773.000000,134542.000000
mean,278047.300091,5.612709,2.532133,0.733236,2.333398,0.016113,0.529658,30.253224
std,102849.568343,4.430354,4.158891,1.367993,7.724338,0.049311,0.942016,17.466661
min,100001.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,188878.500000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,17.000000
50%,277895.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,28.000000
75%,367184.500000,8.000000,4.000000,1.000000,2.000000,0.010000,1.000000,40.888889
max,456255.000000,116.000000,116.000000,38.000000,396.000000,1.000000,5.000000,97.000000


In [102]:
bb_applicant_features["bureau_bb_history_coverage"] = (
    bb_applicant_features["bureau_bb_accounts_with_history"] /
    bb_applicant_features["bureau_bb_accounts"]
)

In [103]:

delinquent_accounts = bb_applicant_features[
    "bureau_bb_delinquent_accounts"
]

bb_applicant_features["bureau_bb_has_delinquency"] = (
    delinquent_accounts
    .gt(0)
    .where(delinquent_accounts.notna())
    .astype("Int8")
)

| Column | English explanation | المعنى بالعربي |
|---|---|---|
| `bureau_bb_accounts_with_history` | Number of external accounts with available monthly history. | عدد الحسابات التي يتوفر لها سجل شهري. |
| `bureau_bb_history_coverage` | Proportion of external accounts that have available monthly history. | نسبة الحسابات التي يتوفر لها سجل شهري إلى إجمالي الحسابات. |
| `bureau_bb_has_delinquency` | Binary indicator: `1` if the applicant has at least one account with observed delinquency, otherwise `0`. | مؤشر: 1 إذا وُجد تأخير مسجل في أي حساب، و0 إذا لم يُسجل تأخير في السجلات المتاحة. |

In [104]:
bb_applicant_features[
    [
        "bureau_bb_accounts",
        "bureau_bb_accounts_with_history",
        "bureau_bb_history_coverage",
        "bureau_bb_delinquent_accounts",
        "bureau_bb_has_delinquency"
    ]
].describe()

,bureau_bb_accounts,bureau_bb_accounts_with_history,bureau_bb_history_coverage,bureau_bb_delinquent_accounts,bureau_bb_has_delinquency
count,305811.000000,305811.000000,305811.000000,134542.000000,134542.0
mean,5.612709,2.532133,0.439707,0.733236,0.360646
std,4.430354,4.158891,0.496178,1.367993,0.48019
min,1.000000,0.000000,0.000000,0.000000,0.0
25%,2.000000,0.000000,0.000000,0.000000,0.0
50%,4.000000,0.000000,0.000000,0.000000,0.0
75%,8.000000,4.000000,1.000000,1.000000,1.0
max,116.000000,116.000000,1.000000,38.000000,1.0


In [105]:
bureau_features = bureau_features.merge(
    bb_applicant_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

In [106]:
print("Shape:", bureau_features.shape)

print(
    "Unique applicants:",
    bureau_features["SK_ID_CURR"].nunique()
)

bureau_features.head()

Shape: (305811, 32)
Unique applicants: 305811


,SK_ID_CURR,bureau_total_accounts,bureau_active_accounts,bureau_closed_accounts,bureau_consumer_credit_accounts,bureau_credit_card_accounts,bureau_credit_sum_total,bureau_credit_sum_mean,bureau_credit_sum_max,bureau_debt_total,...,bureau_accounts_with_debt_ratio,bureau_bb_accounts,bureau_bb_accounts_with_history,bureau_bb_delinquent_accounts,bureau_bb_delinquent_months,bureau_bb_delinquent_rate_mean,bureau_bb_max_status,bureau_bb_months_mean,bureau_bb_history_coverage,bureau_bb_has_delinquency
0,100001,7,3,4,7,0,1453365.000,207623.571429,378000.0,596686.5,...,0.428571,7,7,1.0,1.0,0.007519,1.0,24.571429,1.0,1
1,100002,8,2,6,4,4,865055.565,108131.945625,450000.0,245781.0,...,0.125000,8,8,6.0,27.0,0.255682,1.0,13.750000,1.0,1
2,100003,4,1,3,2,2,1017400.500,254350.125000,810000.0,0.0,...,0.000000,4,0,NaN,NaN,NaN,NaN,NaN,0.0,<NA>
3,100004,2,0,2,2,0,189037.800,94518.900000,94537.8,0.0,...,0.000000,2,0,NaN,NaN,NaN,NaN,NaN,0.0,<NA>
4,100005,3,2,1,2,1,657126.000,219042.000000,568800.0,568408.5,...,0.666667,3,3,0.0,0.0,0.000000,0.0,7.000000,1.0,0


In [107]:
bureau_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 305811 entries, 0 to 305810
Data columns (total 32 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   SK_ID_CURR                       305811 non-null  int64  
 1   bureau_total_accounts            305811 non-null  int64  
 2   bureau_active_accounts           305811 non-null  int64  
 3   bureau_closed_accounts           305811 non-null  int64  
 4   bureau_consumer_credit_accounts  305811 non-null  int64  
 5   bureau_credit_card_accounts      305811 non-null  int64  
 6   bureau_credit_sum_total          305811 non-null  float64
 7   bureau_credit_sum_mean           305809 non-null  float64
 8   bureau_credit_sum_max            305809 non-null  float64
 9   bureau_debt_total                297439 non-null  float64
 10  bureau_debt_mean                 297439 non-null  float64
 11  bureau_debt_max                  297439 non-null  float64
 12  bureau_overdu

In [108]:
bureau["DAYS_CREDIT"].describe()

count    1.716428e+06
mean    -1.142108e+03
std      7.951649e+02
min     -2.922000e+03
25%     -1.666000e+03
50%     -9.870000e+02
75%     -4.740000e+02
max      0.000000e+00
Name: DAYS_CREDIT, dtype: float64

In [109]:
bureau_features[
    [
        "bureau_days_credit_min",
        "bureau_days_credit_max",
        "bureau_days_credit_mean"
    ]
].describe()

,bureau_days_credit_min,bureau_days_credit_max,bureau_days_credit_mean
count,305811.000000,305811.000000,305811.000000
mean,-1764.362845,-490.942608,-1083.802103
std,859.260807,533.529324,558.441124
min,-2922.000000,-2922.000000,-2922.000000
25%,-2583.000000,-623.000000,-1431.000000
50%,-1822.000000,-305.000000,-1052.500000
75%,-1045.000000,-149.000000,-670.369318
max,0.000000,0.000000,0.000000


In [110]:
print("Shape:", bureau_features.shape)
print("Unique applicants:", bureau_features["SK_ID_CURR"].nunique())
print("Duplicate IDs:", bureau_features["SK_ID_CURR"].duplicated().sum())
print("\nMissing values:")
display(bureau_features.isnull().sum().sort_values(ascending=False).head(10))

Shape: (305811, 32)
Unique applicants: 305811
Duplicate IDs: 0

Missing values:


bureau_bb_max_status              175038
bureau_bb_delinquent_rate_mean    171269
bureau_bb_has_delinquency         171269
bureau_bb_months_mean             171269
bureau_bb_delinquent_months       171269
bureau_bb_delinquent_accounts     171269
bureau_debt_mean                    8372
bureau_debt_max                     8372
bureau_debt_total                   8372
bureau_credit_sum_mean                 2
dtype: int64

In [111]:
bureau_features.to_parquet(
    DATA_INTERIM / "bureau_features.parquet",
    index=False
)

In [112]:
saved_bureau = pd.read_parquet(
    DATA_INTERIM / "bureau_features.parquet"
)

print("Saved shape:", saved_bureau.shape)
print("Unique IDs:", saved_bureau["SK_ID_CURR"].is_unique)

Saved shape: (305811, 32)
Unique IDs: True


In [113]:
saved_bureau.head()

,SK_ID_CURR,bureau_total_accounts,bureau_active_accounts,bureau_closed_accounts,bureau_consumer_credit_accounts,bureau_credit_card_accounts,bureau_credit_sum_total,bureau_credit_sum_mean,bureau_credit_sum_max,bureau_debt_total,...,bureau_accounts_with_debt_ratio,bureau_bb_accounts,bureau_bb_accounts_with_history,bureau_bb_delinquent_accounts,bureau_bb_delinquent_months,bureau_bb_delinquent_rate_mean,bureau_bb_max_status,bureau_bb_months_mean,bureau_bb_history_coverage,bureau_bb_has_delinquency
0,100001,7,3,4,7,0,1453365.000,207623.571429,378000.0,596686.5,...,0.428571,7,7,1.0,1.0,0.007519,1.0,24.571429,1.0,1
1,100002,8,2,6,4,4,865055.565,108131.945625,450000.0,245781.0,...,0.125000,8,8,6.0,27.0,0.255682,1.0,13.750000,1.0,1
2,100003,4,1,3,2,2,1017400.500,254350.125000,810000.0,0.0,...,0.000000,4,0,NaN,NaN,NaN,NaN,NaN,0.0,<NA>
3,100004,2,0,2,2,0,189037.800,94518.900000,94537.8,0.0,...,0.000000,2,0,NaN,NaN,NaN,NaN,NaN,0.0,<NA>
4,100005,3,2,1,2,1,657126.000,219042.000000,568800.0,568408.5,...,0.666667,3,3,0.0,0.0,0.000000,0.0,7.000000,1.0,0


In [114]:
saved_bureau.info()

<class 'pandas.DataFrame'>
RangeIndex: 305811 entries, 0 to 305810
Data columns (total 32 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   SK_ID_CURR                       305811 non-null  int64  
 1   bureau_total_accounts            305811 non-null  int64  
 2   bureau_active_accounts           305811 non-null  int64  
 3   bureau_closed_accounts           305811 non-null  int64  
 4   bureau_consumer_credit_accounts  305811 non-null  int64  
 5   bureau_credit_card_accounts      305811 non-null  int64  
 6   bureau_credit_sum_total          305811 non-null  float64
 7   bureau_credit_sum_mean           305809 non-null  float64
 8   bureau_credit_sum_max            305809 non-null  float64
 9   bureau_debt_total                297439 non-null  float64
 10  bureau_debt_mean                 297439 non-null  float64
 11  bureau_debt_max                  297439 non-null  float64
 12  bureau_overdu